<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/19-sequences-and-transformers/03_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 19: Solutions

Try the exercises first. The notes after each solution explain the idea behind it and the mistake each check is designed to catch.

In [1]:
# Setup: run this cell once before you start.
# It defines the check functions used below.

import collections
import math

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


class Raises:
    """Marks a test case where the code is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, torch.Tensor) or isinstance(b, torch.Tensor):
        if not (isinstance(a, torch.Tensor) and isinstance(b, torch.Tensor)):
            return False
        a, b = a.detach().cpu().numpy(), b.detach().cpu().numpy()
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        try:
            a_arr, b_arr = np.asarray(a), np.asarray(b)
        except Exception:
            return False
        if a_arr.shape != b_arr.shape:
            return False
        if a_arr.dtype.kind in "fc" or b_arr.dtype.kind in "fc":
            return bool(np.allclose(a_arr, b_arr, rtol=tol, atol=tol, equal_nan=True))
        return bool(np.array_equal(a_arr, b_arr))
    if isinstance(a, (float, np.floating)) or isinstance(b, (float, np.floating)):
        if not isinstance(a, (int, float, np.number)) or not isinstance(b, (int, float, np.number)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y, tol) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k], tol) for k in a)
    return a == b


def _short(x, limit=90):
    if isinstance(x, torch.Tensor):
        body = np.array2string(x.detach().cpu().numpy(), precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"tensor(shape={tuple(x.shape)}, dtype={str(x.dtype).replace('torch.', '')}, {' '.join(body.split())})"
    elif isinstance(x, np.ndarray):
        body = np.array2string(x, precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"array(shape={x.shape}, dtype={x.dtype}, {' '.join(body.split())})"
    elif isinstance(x, (list, tuple)) and any(isinstance(i, (np.ndarray, torch.Tensor)) for i in x):
        brackets = "()" if isinstance(x, tuple) else "[]"
        return brackets[0] + ",\n           ".join(_short(i, limit) for i in x) + brackets[1]
    else:
        text = repr(x)
    return text if len(text) <= limit else text[:limit - 3] + "..."


def _scenarios(name, cases):
    for description, scenario, expected, hint in cases:
        try:
            result = scenario()
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {name}: {description}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {_short(expected)}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {name}: {description}")
            print(f"      expected {expected!r}, but nothing was raised")
            if hint:
                print(f"      hint: {hint}")
            return
        if not _same(result, expected):
            print(f"FAIL  {name}: {description}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {name}")


# ---- checks ----

def check_encode_chars(f):
    vocab = list("\n !',.:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz")
    line = "To be, or not to be: that is the question."
    _scenarios("encode_chars", [
        ("the example", lambda: f("abba", ["a", "b"]), [0, 1, 1, 0], None),
        ("a line of Shakespeare", lambda: f(line, vocab), [vocab.index(c) for c in line], "a character's number is its position in vocab"),
        ("an unsorted vocabulary", lambda: f("cab", ["c", "a", "b"]), [0, 1, 2], None),
        ("the empty string", lambda: f("", vocab), [], None),
        ("returns a list of ints", lambda: [type(f("ab", ["a", "b"])).__name__] + sorted({type(i).__name__ for i in f("ab", ["a", "b"])}),
         ["list", "int"], None),
        ("an unknown character raises", lambda: f("café", vocab), Raises(ValueError), None),
    ])


def check_causal_mask(f):
    _scenarios("causal_mask", [
        ("the example", lambda: f(3), torch.tensor([[False, True, True], [False, False, True], [False, False, False]]), None),
        ("size 8", lambda: f(8), torch.triu(torch.ones(8, 8, dtype=torch.bool), diagonal=1), "True above the diagonal, False on and below it"),
        ("dtype", lambda: str(f(4).dtype), "torch.bool", None),
        ("a single position can see itself", lambda: f(1), torch.tensor([[False]]), None),
    ])


def check_next_token_pairs(f):
    data = torch.arange(100, 200)
    starts = [0, 17, 63, 89]
    _scenarios("next_token_pairs", [
        ("the example", lambda: f(torch.arange(10), 3, [0, 5]), (torch.tensor([[0, 1, 2], [5, 6, 7]]), torch.tensor([[1, 2, 3], [6, 7, 8]])), None),
        ("four windows", lambda: f(data, 10, starts),
         (torch.stack([data[s:s + 10] for s in starts]), torch.stack([data[s + 1:s + 11] for s in starts])), "y is x shifted by one"),
        ("the last valid start", lambda: f(data, 10, [89])[1][0, -1].item(), 199, None),
        ("a start that's too late raises", lambda: f(data, 10, [90]), Raises(ValueError), "the target needs block_size + 1 tokens"),
        ("a negative start raises", lambda: f(data, 10, [-5]), Raises(ValueError), None),
    ])


def check_temperature_top_k(f):
    g = torch.Generator().manual_seed(0)
    logits = torch.randn(4, 10, generator=g, dtype=torch.float64) * 2

    def reference(x, temperature=1.0, k=None):
        x = x / temperature
        if k is not None:
            kth = x.topk(k, dim=-1).values[..., -1:]
            x = x.masked_fill(x < kth, float("-inf"))
        return torch.softmax(x, dim=-1)

    tied = torch.tensor([3.0, 1.0, 1.0, 0.0], dtype=torch.float64)
    _scenarios("temperature_top_k", [
        ("the example", lambda: f(torch.tensor([2.0, 1.0, 0.0]), k=2), torch.tensor([0.7310586, 0.2689414, 0.0]), None),
        ("temperature 1 is a plain softmax", lambda: f(logits), torch.softmax(logits, dim=-1), None),
        ("temperature 0.5 sharpens", lambda: f(logits, temperature=0.5), reference(logits, 0.5), "divide the logits by the temperature"),
        ("temperature 2 with top-3, a batch of rows", lambda: f(logits, temperature=2.0, k=3), reference(logits, 2.0, 3),
         "keep the k largest in each row, then renormalise"),
        ("ties with the k-th value are kept", lambda: f(tied, k=2), reference(tied, k=2), None),
        ("temperature 0 raises", lambda: f(logits, temperature=0), Raises(ValueError), None),
    ])


def check_bigram_probabilities(f):
    g = torch.Generator().manual_seed(1)
    tokens = torch.randint(0, 6, (500,), generator=g)

    def reference(tok, v, s):
        counts = np.zeros((v, v))
        for a, b in zip(tok[:-1].tolist(), tok[1:].tolist()):
            counts[a, b] += 1
        counts += s
        return torch.tensor(counts / counts.sum(axis=1, keepdims=True), dtype=torch.float32)

    _scenarios("bigram_probabilities", [
        ("the example", lambda: f([0, 1, 0, 1, 1], 2)[0], torch.tensor([0.25, 0.75]), None),
        ("500 random tokens, repeated pairs counted every time", lambda: f(tokens, 6), reference(tokens, 6, 1.0),
         "repeated pairs must each add 1: use index_put_ with accumulate=True, or a loop"),
        ("smoothing 0.1", lambda: f(tokens, 6, smoothing=0.1), reference(tokens, 6, 0.1), None),
        ("a token with no data gets a uniform row", lambda: f([0, 1, 0], 3)[2], torch.full((3,), 1 / 3), None),
        ("rows add up to 1", lambda: f(tokens, 8).sum(dim=1), torch.ones(8), None),
        ("smoothing 0 raises", lambda: f(tokens, 6, smoothing=0), Raises(ValueError), None),
    ])


def check_merge_pair(f):
    original = ["t", "h", "e", " ", "t", "h", "e", "n"]

    def unchanged():
        f(original, ("t", "h"), "th")
        return original

    _scenarios("merge_pair", [
        ("the first example", lambda: f(["l", "o", "w", "l", "o"], ("l", "o"), "lo"), ["lo", "w", "lo"], None),
        ("overlapping occurrences", lambda: f(["a", "a", "a"], ("a", "a"), "aa"), ["aa", "a"], "after a merge, continue after both tokens"),
        ("four in a row", lambda: f(["a", "a", "a", "a"], ("a", "a"), "aa"), ["aa", "aa"], None),
        ("numbers", lambda: f([1, 2, 3, 1, 2], (1, 2), 99), [99, 3, 99], None),
        ("no occurrence", lambda: f(["x", "y"], ("y", "x"), "yx"), ["x", "y"], None),
        ("only the first half matches", lambda: f(["a", "c", "a", "b"], ("a", "b"), "ab"), ["a", "c", "ab"], "both tokens of the pair must match"),
        ("the pair at the very end", lambda: f(["a", "b", "c"], ("b", "c"), "bc"), ["a", "bc"], None),
        ("the input list is not changed", unchanged, ["t", "h", "e", " ", "t", "h", "e", "n"], None),
    ])


def check_scaled_dot_product_attention(f):
    g = torch.Generator().manual_seed(2)
    q = torch.randn(2, 3, 7, 8, generator=g, dtype=torch.float64)
    k = torch.randn(2, 3, 7, 8, generator=g, dtype=torch.float64)
    v = torch.randn(2, 3, 7, 5, generator=g, dtype=torch.float64)
    k_long = torch.randn(2, 3, 9, 8, generator=g, dtype=torch.float64)
    v_long = torch.randn(2, 3, 9, 5, generator=g, dtype=torch.float64)

    def weights_of(causal):
        return f(q, k, v, causal=causal)[1]

    _scenarios("scaled_dot_product_attention", [
        ("causal output", lambda: f(q, k, v)[0], F.scaled_dot_product_attention(q, k, v, is_causal=True),
         "scale by sqrt of the query size, and mask the future"),
        ("weights are zero above the diagonal", lambda: weights_of(True).triu(diagonal=1).abs().max().item(), 0.0, None),
        ("weights add up to 1", lambda: weights_of(True).sum(dim=-1), torch.ones(2, 3, 7, dtype=torch.float64), None),
        ("without the mask", lambda: f(q, k, v, causal=False)[0], F.scaled_dot_product_attention(q, k, v), None),
        ("cross-attention: 7 queries, 9 keys", lambda: f(q, k_long, v_long, causal=False)[0],
         F.scaled_dot_product_attention(q, k_long, v_long), "use transpose(-2, -1), not .T"),
        ("a single sequence without batch dimensions", lambda: f(q[0, 0], k[0, 0], v[0, 0])[0],
         F.scaled_dot_product_attention(q[0, 0][None], k[0, 0][None], v[0, 0][None], is_causal=True)[0], None),
    ])


def check_sinusoidal_positions(f):
    def reference(n, d):
        pe = np.zeros((n, d))
        for pos in range(n):
            for i in range(d // 2):
                angle = pos / 10000 ** (2 * i / d)
                pe[pos, 2 * i] = math.sin(angle)
                pe[pos, 2 * i + 1] = math.cos(angle)
        return torch.tensor(pe, dtype=torch.float32)

    _scenarios("sinusoidal_positions", [
        ("row 0", lambda: f(4, 6)[0], torch.tensor([0.0, 1, 0, 1, 0, 1]), None),
        ("a small table", lambda: f(10, 8), reference(10, 8), "the exponent is 2 * i / d_model, and sines and cosines alternate"),
        ("a larger table", lambda: f(64, 32), reference(64, 32), None),
        ("dtype and shape", lambda: (str(f(5, 4).dtype), tuple(f(5, 4).shape)), ("torch.float32", (5, 4)), None),
        ("an odd size raises", lambda: f(5, 7), Raises(ValueError), None),
    ])


class _NextPlusOne(nn.Module):
    """Predicts (token + 1) mod 10 at every position; refuses inputs longer than 6."""

    def __init__(self):
        super().__init__()
        self.dummy = nn.Linear(1, 1)

    def forward(self, idx):
        if idx.shape[1] > 6:
            raise RuntimeError("input longer than the block size")
        return F.one_hot((idx + 1) % 10, 10).float() * 5


class _OwnPlusThree(nn.Module):
    """At every position, predicts that position's own token plus 3."""

    def forward(self, idx):
        return F.one_hot((idx + 3) % 10, 10).float()


class _ModeSpy(nn.Module):
    def __init__(self):
        super().__init__()
        self.drop = nn.Dropout(0.5)
        self.modes, self.grads = [], []

    def forward(self, idx):
        self.modes.append(self.training)
        self.grads.append(torch.is_grad_enabled())
        return F.one_hot(idx % 10, 10).float()


def check_generate_greedy(f):
    spy = _ModeSpy()

    def spied():
        f(spy, torch.tensor([[1, 2]]), 3, 6)
        return (all(not m for m in spy.modes), any(spy.grads))

    _scenarios("generate_greedy", [
        ("counts upwards", lambda: f(_NextPlusOne(), torch.tensor([[3]]), 5, 6), torch.tensor([[3, 4, 5, 6, 7, 8]]), None),
        ("crops the context to block_size", lambda: f(_NextPlusOne(), torch.tensor([[0, 1]]), 10, 6)[0, -1].item(), 1,
         "give the model only the last block_size tokens"),
        ("uses the prediction at the last position", lambda: f(_OwnPlusThree(), torch.tensor([[1, 2, 3]]), 1, 6),
         torch.tensor([[1, 2, 3, 6]]), "logits[:, -1, :] is the prediction for the next token"),
        ("a batch of two", lambda: f(_NextPlusOne(), torch.tensor([[1], [7]]), 3, 6), torch.tensor([[1, 2, 3, 4], [7, 8, 9, 0]]), None),
        ("evaluation mode, no gradients", spied, (True, False), "call model.eval() and use torch.no_grad()"),
    ])


def _reference_bpe(word_counts, n_merges):
    split = {w: list(w) for w in word_counts}
    merges = []
    for _ in range(n_merges):
        pairs = collections.Counter()
        for w, n in word_counts.items():
            p = split[w]
            for pair in zip(p, p[1:]):
                pairs[pair] += n
        if not pairs:
            break
        best = max(pairs, key=lambda q: (pairs[q], q))
        merges.append(best)
        for w in split:
            p, out, i = split[w], [], 0
            while i < len(p):
                if i + 1 < len(p) and (p[i], p[i + 1]) == best:
                    out.append(p[i] + p[i + 1])
                    i += 2
                else:
                    out.append(p[i])
                    i += 1
            split[w] = out
    return merges


def check_bpe_train(f):
    classic = {"low": 5, "lower": 2, "newest": 6, "widest": 3}
    sample = ("the quick brown fox jumps over the lazy dog and the dog barks back at the fox "
              "while the other foxes and dogs watch the show from the hill over there ") * 3
    sample_counts = dict(collections.Counter(" " + w for w in sample.split()))
    _scenarios("bpe_train", [
        ("the first two merges", lambda: f(classic, 2), [("s", "t"), ("e", "st")], "ties go to the larger pair: max by (count, pair)"),
        ("ten merges", lambda: f(classic, 10), _reference_bpe(classic, 10), None),
        ("counts weight the pairs", lambda: f({"ab": 1, "cd": 5}, 1), [("c", "d")], "add the word's count, not 1"),
        ("stops when every word is one token", lambda: f({"ab": 3, "c": 1}, 5), [("a", "b")], None),
        ("overlaps inside a word", lambda: f({"aaaa": 1, "aaa": 2}, 3), _reference_bpe({"aaaa": 1, "aaa": 2}, 3), None),
        ("a larger sample", lambda: f(sample_counts, 30), _reference_bpe(sample_counts, 30), None),
    ])


def _causal_bool(t):
    return torch.triu(torch.ones(t, t, dtype=torch.bool), diagonal=1)


def _torch_mha(d, h):
    torch.manual_seed(0)
    mha = nn.MultiheadAttention(d, h, batch_first=True).double()
    with torch.no_grad():
        mha.in_proj_bias.normal_(0, 0.2)
        mha.out_proj.bias.normal_(0, 0.2)
    return mha


def check_MultiHeadAttention(cls):
    x = torch.randn(3, 7, 16, generator=torch.Generator().manual_seed(3), dtype=torch.float64)

    def yours(d, h):
        reference = _torch_mha(d, h)
        mine = cls(d, h).double()
        with torch.no_grad():
            mine.qkv.weight.copy_(reference.in_proj_weight)
            mine.qkv.bias.copy_(reference.in_proj_bias)
            mine.proj.weight.copy_(reference.out_proj.weight)
            mine.proj.bias.copy_(reference.out_proj.bias)
        return mine(x[..., :d])

    def theirs(d, h):
        reference = _torch_mha(d, h)
        out, _ = reference(x[..., :d], x[..., :d], x[..., :d], attn_mask=_causal_bool(7), need_weights=False)
        return out.detach()

    def future_leak():
        mine = cls(16, 4).double()
        a = x.clone()
        b = x.clone()
        b[:, 5:] = 0
        with torch.no_grad():
            return (mine(a)[:, :5] - mine(b)[:, :5]).abs().max().item()

    _scenarios("MultiHeadAttention", [
        ("output shape", lambda: tuple(cls(16, 4).double()(x).shape), (3, 7, 16), None),
        ("parameters", lambda: sum(p.numel() for p in cls(16, 4).parameters()), 4 * 16 * 16 + 4 * 16, None),
        ("the future can't affect the past", future_leak, 0.0, "apply the causal mask"),
        ("matches nn.MultiheadAttention, 4 heads", lambda: yours(16, 4), theirs(16, 4),
         "view as (b, t, heads, head_size), then transpose(1, 2)"),
        ("matches nn.MultiheadAttention, 2 heads", lambda: yours(16, 2), theirs(16, 2), None),
        ("matches nn.MultiheadAttention, 1 head", lambda: yours(8, 1), theirs(8, 1), None),
        ("d_model not divisible by n_heads raises", lambda: cls(10, 4), Raises(ValueError), None),
    ])


def _torch_block(d, h):
    torch.manual_seed(0)
    layer = nn.TransformerEncoderLayer(d, h, dim_feedforward=4 * d, dropout=0.0, activation="gelu",
                                       batch_first=True, norm_first=True).double()
    with torch.no_grad():
        for p in layer.parameters():
            if p.dim() == 1:
                p.normal_(0, 0.2)
        layer.norm1.weight.add_(1.0)
        layer.norm2.weight.add_(1.0)
    return layer


def check_TransformerBlock(cls):
    x = torch.randn(2, 6, 16, generator=torch.Generator().manual_seed(4), dtype=torch.float64)

    def yours(d, h):
        ref = _torch_block(d, h)
        mine = cls(d, h).double()
        with torch.no_grad():
            mine.ln1.weight.copy_(ref.norm1.weight)
            mine.ln1.bias.copy_(ref.norm1.bias)
            mine.ln2.weight.copy_(ref.norm2.weight)
            mine.ln2.bias.copy_(ref.norm2.bias)
            mine.attn.qkv.weight.copy_(ref.self_attn.in_proj_weight)
            mine.attn.qkv.bias.copy_(ref.self_attn.in_proj_bias)
            mine.attn.proj.weight.copy_(ref.self_attn.out_proj.weight)
            mine.attn.proj.bias.copy_(ref.self_attn.out_proj.bias)
            mine.mlp[0].weight.copy_(ref.linear1.weight)
            mine.mlp[0].bias.copy_(ref.linear1.bias)
            mine.mlp[2].weight.copy_(ref.linear2.weight)
            mine.mlp[2].bias.copy_(ref.linear2.bias)
        return mine(x[..., :d])

    def theirs(d, h):
        ref = _torch_block(d, h)
        ref.train()                                         # avoids PyTorch's separate fast path; no dropout anyway
        return ref(x[..., :d], src_mask=_causal_bool(6)).detach()

    _scenarios("TransformerBlock", [
        ("output shape", lambda: tuple(cls(16, 4).double()(x).shape), (2, 6, 16), None),
        ("parameters", lambda: sum(p.numel() for p in cls(16, 4).parameters()),
         sum(p.numel() for p in _torch_block(16, 4).parameters()), "the MLP is 4 times wider than d_model"),
        ("layer types", lambda: [type(m).__name__ for m in cls(16, 4).mlp], ["Linear", "GELU", "Linear"], None),
        ("matches nn.TransformerEncoderLayer, 4 heads", lambda: yours(16, 4), theirs(16, 4),
         "pre-norm: x = x + attn(ln1(x)), then x = x + mlp(ln2(x))"),
        ("matches nn.TransformerEncoderLayer, 2 heads", lambda: yours(8, 2), theirs(8, 2), None),
    ])


print("Setup done. Checks are ready.")

Setup done. Checks are ready.


## Easy

### Exercise 1: `encode_chars`

In [2]:
def encode_chars(text: str, vocab) -> list:
    lookup = {ch: i for i, ch in enumerate(vocab)}
    ids = []
    for ch in text:
        if ch not in lookup:
            raise ValueError(f"character {ch!r} is not in the vocabulary")
        ids.append(lookup[ch])
    return ids

In [3]:
check_encode_chars(encode_chars)

PASS  all 6 tests passed for encode_chars


Building a dictionary first makes each lookup instant. `vocab.index(ch)` gives the same answers, but it searches the list from the start every time, so encoding a million characters with a vocabulary of 65 does tens of millions of comparisons. For real tokenisers with 100,000 tokens the difference is enormous.

The `ValueError` is the weakness of character and word vocabularies that BPE fixes: a vocabulary learnt from Shakespeare has no `"#"` or `"é"`, so text containing them can't be encoded at all. Byte-level BPE, used by GPT models, starts from the 256 possible bytes, so every possible text can be encoded.

Saying **which** character failed in the error message costs one line and saves a lot of time when it happens on a large file.

### Exercise 2: `causal_mask`

In [4]:
def causal_mask(t: int) -> torch.Tensor:
    return torch.triu(torch.ones(t, t, dtype=torch.bool), diagonal=1)

In [5]:
check_causal_mask(causal_mask)

PASS  all 4 tests passed for causal_mask


`torch.triu` keeps the upper triangle of a matrix and zeroes the rest; `diagonal=1` starts one place above the main diagonal, so each position can still look at itself. With `diagonal=0` a position couldn't see its own token, and the very first position would have nothing at all to attend to: its row of scores would be all minus infinity, and the softmax would give `nan`.

Libraries disagree about what `True` means in a mask. Here, as in `masked_fill` and in `nn.MultiheadAttention`'s `attn_mask`, `True` means "blocked". In `F.scaled_dot_product_attention`'s `attn_mask`, a boolean `True` means "allowed". It's worth checking every time; a mask with the wrong meaning lets the model see the future, and the training loss drops suspiciously fast while generation produces garbage.

### Exercise 3: `next_token_pairs`

In [6]:
def next_token_pairs(data: torch.Tensor, block_size: int, starts):
    for s in starts:
        if s < 0 or s + block_size + 1 > len(data):
            raise ValueError(f"start {s} leaves too little data for a block of {block_size}")
    x = torch.stack([data[s:s + block_size] for s in starts])
    y = torch.stack([data[s + 1:s + block_size + 1] for s in starts])
    return x, y

In [7]:
check_next_token_pairs(next_token_pairs)

PASS  all 5 tests passed for next_token_pairs


The target is the input shifted by one, so every position in the window is a training example: from `x[:, :1]` predict `y[:, 0]`, from `x[:, :2]` predict `y[:, 1]`, and so on. One window of 64 characters gives 64 predictions, each with a different amount of context, and the causal mask makes sure each prediction only uses the positions before it.

The bounds check matters because slicing a tensor past its end doesn't raise an error in Python: it quietly returns a shorter slice. `torch.stack` then fails with a confusing message about sizes, or worse, if every slice is short by the same amount, you get a batch that's subtly too short. The last valid start is `len(data) - block_size - 1`, which is why the lesson draws starts with `torch.randint(len(source) - block_size - 1, ...)`.

### Exercise 4: `temperature_top_k`

In [8]:
def temperature_top_k(logits: torch.Tensor, temperature: float = 1.0, k=None) -> torch.Tensor:
    if temperature <= 0:
        raise ValueError("temperature must be positive")
    logits = logits / temperature
    if k is not None:
        cutoff = logits.topk(k, dim=-1).values[..., -1:]
        logits = logits.masked_fill(logits < cutoff, float("-inf"))
    return torch.softmax(logits, dim=-1)

In [9]:
check_temperature_top_k(temperature_top_k)

PASS  all 6 tests passed for temperature_top_k


Setting the dropped logits to minus infinity before the softmax makes their probabilities exactly 0 and renormalises the rest in one step. Zeroing probabilities **after** the softmax would leave the kept ones adding up to less than 1, and `torch.multinomial` would still sample correctly (it renormalises), but anything else that uses the probabilities would be off.

`values[..., -1:]` is the `k`-th largest value of each row, with the last dimension kept so it broadcasts against the row. The `...` makes the same code work for one vector or a batch.

The order of the two steps doesn't change which tokens are kept, since dividing by a positive temperature keeps the order of the logits. It does change the kept probabilities, and the definition used in practice applies the temperature first.

Why is temperature 0 an error rather than "pick the most likely token"? Dividing by 0 gives infinities and `nan`. Code that wants greedy decoding should call `argmax` directly; Exercise 9 does exactly that.

## Medium

### Exercise 5: `bigram_probabilities`

In [10]:
def bigram_probabilities(tokens, vocab_size: int, smoothing: float = 1.0) -> torch.Tensor:
    if smoothing <= 0:
        raise ValueError("smoothing must be positive")
    tokens = torch.as_tensor(tokens, dtype=torch.long)
    counts = torch.zeros(vocab_size, vocab_size)
    counts.index_put_((tokens[:-1], tokens[1:]), torch.ones(len(tokens) - 1), accumulate=True)
    counts += smoothing
    return counts / counts.sum(dim=1, keepdim=True)

In [11]:
check_bigram_probabilities(bigram_probabilities)

PASS  all 6 tests passed for bigram_probabilities


`index_put_` with `accumulate=True` adds 1 at every `(previous, next)` position in one go. The tempting shortcut `counts[tokens[:-1], tokens[1:]] += 1` looks equivalent but isn't: when the same pair appears several times, the repeated indices all write to the same cell and it's only incremented **once**. One check uses a text with repeated pairs to catch that. A Python loop over the pairs also works, just more slowly.

**Smoothing** (here, add-one or Laplace smoothing) is there because the validation text will contain pairs that never appeared in training. With a probability of 0, `-log(0)` is infinite, and a single unseen pair would make the whole loss infinite. Adding a small count everywhere keeps every probability positive. It's a crude fix; a neural model gets the same benefit more gracefully, because similar tokens get similar embeddings.

Rows with no data at all (a token that never appears before another) become uniform, which is a sensible answer for "no idea".

### Exercise 6: `merge_pair`

In [12]:
def merge_pair(tokens, pair, new_token) -> list:
    out, i = [], 0
    while i < len(tokens):
        if i + 1 < len(tokens) and tokens[i] == pair[0] and tokens[i + 1] == pair[1]:
            out.append(new_token)
            i += 2
        else:
            out.append(tokens[i])
            i += 1
    return out

In [13]:
check_merge_pair(merge_pair)

PASS  all 8 tests passed for merge_pair


A `while` loop is the natural fit, because the step size changes: 2 after a merge, 1 otherwise. A `for` loop over positions can't skip the second token of a merged pair, so `["a", "a", "a"]` would come out as `["aa", "aa"]`, inventing an extra token.

Overlaps like `"aaa"` are exactly where tokenisers have to make a choice, and left to right is the standard one. Real tokenisers must make the same choice when they **apply** the merges to new text as when they learnt them, or the same word would be tokenised differently in training and in use.

Joining strings to make the new token (`"l" + "o"`) is convenient for reading, but real tokenisers work with token numbers: each merge creates a new number, and a table records which pair it stands for. That's why this function takes `new_token` as an argument instead of building it.

### Exercise 7: `scaled_dot_product_attention`

In [14]:
def scaled_dot_product_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, causal: bool = True):
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])
    if causal:
        t = q.shape[-2]
        future = torch.triu(torch.ones(t, t, dtype=torch.bool, device=q.device), diagonal=1)
        scores = scores.masked_fill(future, float("-inf"))
    weights = torch.softmax(scores, dim=-1)
    return weights @ v, weights

In [15]:
check_scaled_dot_product_attention(scaled_dot_product_attention)

PASS  all 6 tests passed for scaled_dot_product_attention


`k.transpose(-2, -1)` swaps the last two dimensions, whatever comes before them, so the same code works for `(T, d)`, `(batch, T, d)` and `(batch, heads, T, d)`. `k.T` would reverse **all** dimensions, which is only right for a plain matrix, and breaks as soon as there's a batch.

The mask is a `(T, T)` matrix, and broadcasting applies it to every batch and head at once.

**Without the mask** this is the attention used in encoders such as BERT, where every position may see the whole sentence, and in **cross-attention**, where the queries come from one sequence and the keys and values from another, so `T_q` and `T_k` can differ: in translation, each word being written looks at every word of the original sentence. One check uses different lengths for that case.

The scale uses `d`, the size of the queries and keys, not `d_v`; a check with `d_v` different from `d` catches the mix-up.

### Exercise 8: `sinusoidal_positions`

In [16]:
def sinusoidal_positions(n_positions: int, d_model: int) -> torch.Tensor:
    if d_model % 2:
        raise ValueError("d_model must be even")
    pos = torch.arange(n_positions, dtype=torch.float64).unsqueeze(1)          # (n, 1)
    i = torch.arange(0, d_model, 2, dtype=torch.float64)                       # 0, 2, 4, ...
    angles = pos / 10000 ** (i / d_model)                                      # (n, d/2)
    pe = torch.zeros(n_positions, d_model, dtype=torch.float64)
    pe[:, 0::2] = torch.sin(angles)
    pe[:, 1::2] = torch.cos(angles)
    return pe.float()

In [17]:
check_sinusoidal_positions(sinusoidal_positions)

PASS  all 5 tests passed for sinusoidal_positions


Each pair of columns is a sine and cosine wave with its own frequency. The first pair changes quickly (a full cycle every 6 positions or so); the last changes very slowly (a cycle every tens of thousands of positions). Together they give every position a unique pattern, a bit like the hands of a clock with a seconds hand, a minutes hand and an hours hand.

Broadcasting does the work: positions as a column `(n, 1)` divided by frequencies as a row `(d/2,)` gives all the angles as an `(n, d/2)` table. `pe[:, 0::2]` selects the even columns and `pe[:, 1::2]` the odd ones. Computing in `float64` and converting at the end keeps the angles accurate for large positions.

The common slip is using `i` (0, 1, 2, ...) where the formula has `2 * i` in the exponent, which makes the frequencies fall off twice as fast. Another is putting all the sines first and all the cosines after, instead of alternating.

The original paper chose this design partly because the encoding of `pos + k` is a fixed linear function of the encoding of `pos`, which may help attention find relative positions. Learnt position tables (GPT-2) and later **rotary embeddings** (most large models today) have largely replaced it, but the idea of encoding position with waves of different frequencies lives on in rotary embeddings.

### Exercise 9: `generate_greedy`

In [18]:
@torch.no_grad()
def generate_greedy(model: nn.Module, idx: torch.Tensor, n_new: int, block_size: int) -> torch.Tensor:
    model.eval()
    for _ in range(n_new):
        logits = model(idx[:, -block_size:])
        next_id = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        idx = torch.cat([idx, next_id], dim=1)
    return idx

In [19]:
check_generate_greedy(generate_greedy)

PASS  all 5 tests passed for generate_greedy


`idx[:, -block_size:]` takes the last `block_size` tokens, or the whole sequence if it's shorter, without any `if`. Cropping is needed because the position table only has `block_size` rows: position 64 simply doesn't exist for the lesson's model. Cropping from the **front** keeps the most recent context, which matters most for predicting what comes next.

The model returns logits for **every** position, but only the last one is a prediction about the new token; the others are predictions about tokens we already have. `logits[:, -1, :]` picks it. Taking position 0 instead is a classic bug, and one of the checks uses a model whose answer depends on which position you read.

`keepdim=True` keeps the result as a `(batch, 1)` column so it can be joined onto `idx` with `torch.cat` along the time dimension.

Greedy decoding is deterministic, which is useful for testing and for tasks with one right answer (the mini project uses it for arithmetic). For open-ended text it tends to fall into loops, repeating the same phrase, which is why chatbots sample with a temperature instead.

This loop runs the whole model on the whole context for every new token, repeating almost all the work of the previous step. Real systems keep the keys and values from earlier steps in a **KV cache**, so each new token only needs the computation for one new position.

## Hard

### Exercise 10: `bpe_train`

In [20]:
def bpe_train(word_counts: dict, n_merges: int) -> list:
    split = {w: list(w) for w in word_counts}
    merges = []
    for _ in range(n_merges):
        pairs = {}
        for w, n in word_counts.items():
            pieces = split[w]
            for pair in zip(pieces, pieces[1:]):
                pairs[pair] = pairs.get(pair, 0) + n
        if not pairs:
            break
        best = max(pairs, key=lambda p: (pairs[p], p))
        merges.append(best)
        for w in split:
            split[w] = merge_pair(split[w], best, best[0] + best[1])
    return merges

In [21]:
check_bpe_train(bpe_train)

PASS  all 6 tests passed for bpe_train


Working on distinct words with counts is what makes BPE practical. Tiny Shakespeare has over 200,000 words but only about 15,000 distinct ones, so each merge round touches 15,000 short lists instead of a million characters. Counting the pairs **weighted** by the word counts gives exactly the same answer as scanning the whole text, as long as merges never cross word boundaries.

Notice how merges build on each other: `("s", "t")` creates `"st"`, which can then merge with `"e"` into `"est"`. After enough rounds, a common word like `"newest"` becomes a single token, while a rare word stays in pieces.

The tie-breaking rule seems fussy, but it matters: without one, the merges (and therefore every token number) can depend on the order of a dictionary or on details of the implementation, and two programs that "implement BPE" produce different tokenisers. Real tokenisers fix such details so that encoding is exactly reproducible.

The counting is redone from scratch every round, which is simple but wasteful: a merge only changes the counts of pairs next to the merged one. Production tokenisers (written in Rust or C++) update only those counts and keep the pairs in a priority queue, which is how they learn 100,000 merges on gigabytes of text.

### Exercise 11: `MultiHeadAttention`

In [22]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        if d_model % n_heads:
            raise ValueError("d_model must be divisible by n_heads")
        self.n_heads = n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        b, t, d = x.shape
        q, k, v = self.qkv(x).split(d, dim=2)
        q, k, v = (z.view(b, t, self.n_heads, d // self.n_heads).transpose(1, 2) for z in (q, k, v))
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(out.transpose(1, 2).reshape(b, t, d))

In [23]:
check_MultiHeadAttention(MultiHeadAttention)

PASS  all 7 tests passed for MultiHeadAttention


The reshapes are where multi-head attention usually goes wrong, so it's worth being precise:

- `view(b, t, n_heads, head_size)` cuts each position's vector into consecutive slices, one per head: numbers 0 to 31 for head 1, 32 to 63 for head 2, and so on (with 128 numbers and 4 heads).
- `transpose(1, 2)` moves the head dimension in front of the time dimension, giving `(b, heads, t, head_size)`, so that attention treats each head like a separate item in the batch and compares positions within each head.
- After attention, `transpose(1, 2)` and `reshape(b, t, d)` undo this, placing the heads' outputs side by side again.

The tempting shortcut `view(b, n_heads, t, head_size)` (without the transpose) has the right shape but mixes up positions and heads, so "head 2" actually gets slices of different positions. The output has the right shape and the code runs; only the comparison with PyTorch shows it's wrong. That's why the check copies weights into a known-correct implementation.

`reshape` is needed after the transpose (rather than `view`) because the transposed tensor's numbers are no longer stored in order in memory; `reshape` copies them if necessary.

Computing all queries, keys and values with one `Linear` of size `3 * d_model` is the same as three separate layers, just faster, and it's how PyTorch's `nn.MultiheadAttention` stores them too (`in_proj_weight`), which is what makes the weight copy possible.

### Exercise 12: `TransformerBlock`

In [24]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

In [25]:
check_TransformerBlock(TransformerBlock)

PASS  all 5 tests passed for TransformerBlock


Two lines of `forward` contain the whole design. Each part reads a **normalised** copy of `x` and adds its result back to the **original** `x`. The residual stream (the `x` passed from block to block) is never normalised itself, so information and gradients can flow straight through all the blocks.

The checks catch the common variations:

- **Post-norm** (`x = ln1(x + attn(x))`), the arrangement in the original 2017 paper. It's a legitimate design, but it's a different function, and deep post-norm networks are harder to train without careful warm-up, which is why GPT-2 and most models since use pre-norm.
- **Normalising the residual stream**, `x = ln1(x)` followed by `x = x + attn(x)`. This throws away the scale of the residual stream at every block.
- **A ReLU instead of GELU**, or an MLP that isn't 4 times wider. Both still train, but they aren't this block.
- **Forgetting a residual connection**, which turns the block into a plain stack of layers.

PyTorch's `nn.TransformerEncoderLayer` with `norm_first=True` is the same pre-norm block, written as part of PyTorch, which is why the check can copy its weights into yours: `norm1` and `norm2` become `ln1` and `ln2`, `self_attn` becomes `attn`, and `linear1` and `linear2` become the two layers of `mlp`. In your own projects, either is fine; writing it yourself makes every step visible.

## What's next

The mini project in `04_mini_project.ipynb` trains the lesson's GPT on a task with exact answers: adding two 5-digit numbers. You'll find out how much the way the numbers are written matters, test the model on sums it has never seen, look at the mistakes, and see what its attention heads have learnt to do.